# 09. 연도별 게임 출시 수 추이

**분석 목적:** `steamspy_indie_games.csv`의 `release_date`를 기준으로 연도별 Steam 인디게임 출시 수를 확인한다.

**활용 관점:** 출시 게임 수가 얼마나 많은지 파악해, 출시 후 유저 반응을 확보하기 어려운 시장 맥락을 설명한다.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 50)

## 1. 데이터 로드

`data/raw/steamspy_indie_games.csv`를 사용한다. 이 파일은 SteamSpy에서 수집한 인디게임 원본 데이터이며, 출시일은 `release_date` 컬럼에 저장되어 있다.

In [2]:
DATA_PATH = Path("../../data/raw/steamspy_indie_games.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "steamspy_indie_games.csv를 찾을 수 없습니다. "
        "data/raw/steamspy_indie_games.csv 경로를 확인하세요."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df):,}, Columns: {df.shape[1]:,}")

df.head()

Loaded: ../../data/raw/steamspy_indie_games.csv
Rows: 61,266, Columns: 12


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1002,Rag Doll Kung Fu,"20,000 .. 50,000",91,30,99,0,Rag Doll Kung Fu,game,['Indie'],"12 Oct, 2005",Mark Healey
1,1500,Darwinia,"0 .. 20,000",864,216,1199,2,Darwinia,game,"['Indie', 'Strategy']","1 Dec, 2005",Introversion Software
2,1510,Uplink,"500,000 .. 1,000,000",2143,216,1199,2,Uplink,game,"['Indie', 'Strategy']","23 Aug, 2006",Introversion Software
3,1520,DEFCON,"200,000 .. 500,000",3657,525,1199,8,DEFCON,game,"['Indie', 'Strategy']","29 Sep, 2006",Introversion Software
4,1530,Multiwinia,"50,000 .. 100,000",478,107,1199,1,Multiwinia,game,['Indie'],"19 Sep, 2008",Introversion Software


## 2. 출시일 파싱 및 연도 컬럼 생성

SteamSpy의 `release_date`를 날짜형으로 변환한다. 파싱되지 않는 값과 노트북 실행일 기준 미래 출시일은 제외해 실제 출시 수 추이를 집계한다.

In [3]:
release_df = df.copy()
release_df["release_date"] = pd.to_datetime(release_df["release_date"], errors="coerce")

analysis_date = pd.Timestamp.today().normalize()
invalid_release_count = release_df["release_date"].isna().sum()
future_release_count = (release_df["release_date"] > analysis_date).sum()

release_df = release_df.dropna(subset=["release_date"]).copy()
release_df = release_df[release_df["release_date"] <= analysis_date].copy()
release_df["release_year"] = release_df["release_date"].dt.year.astype(int)

print(f"분석 기준일: {analysis_date.date()}")
print(f"출시일 파싱 실패/결측 행 수: {invalid_release_count:,}")
print(f"미래 출시일 제외 행 수: {future_release_count:,}")
print(f"분석 대상 출시 연도 범위: {release_df['release_year'].min()}~{release_df['release_year'].max()}")

분석 기준일: 2026-05-09
출시일 파싱 실패/결측 행 수: 157
미래 출시일 제외 행 수: 1
분석 대상 출시 연도 범위: 1997~2026


## 3. 연도별 출시 수 집계

시계열 그래프는 분석 범위를 2020년부터 2025년까지로 제한한다.

In [4]:
START_YEAR = 2020
END_YEAR = 2025

yearly_release = (
    release_df.query("@START_YEAR <= release_year <= @END_YEAR")
    .groupby("release_year", as_index=False)
    .agg(game_count=("appid", "nunique"))
    .sort_values("release_year")
)

yearly_release["yoy_change"] = yearly_release["game_count"].diff()
yearly_release["yoy_growth_rate"] = yearly_release["game_count"].pct_change() * 100

yearly_release

,release_year,game_count,yoy_change,yoy_growth_rate
0,2020,6381,NaN,NaN
1,2021,6306,-75.0,-1.175364
2,2022,6151,-155.0,-2.457977
3,2023,6779,628.0,10.209722
4,2024,8855,2076.0,30.623986
5,2025,4203,-4652.0,-52.535291


## 4. 시계열 그래프

라인 차트는 2020년부터 2025년까지의 출시 수 추세를 보기 쉽고, 마커는 특정 연도의 출시 수를 직접 확인하기 좋다.

In [5]:
fig = px.line(
    yearly_release,
    x="release_year",
    y="game_count",
    markers=True,
    title="연도별 Steam 인디게임 출시 수 추이 (2020~2025)",
    labels={
        "release_year": "출시 연도",
        "game_count": "게임 출시 수",
    },
    hover_data={
        "release_year": True,
        "game_count": ":,",
        "yoy_change": ":,.0f",
        "yoy_growth_rate": ":.1f",
    },
)

fig.update_traces(line_width=3, marker_size=7)
fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    width=1000,
    height=520,
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(START_YEAR, END_YEAR + 1)),
        range=[START_YEAR - 0.2, END_YEAR + 0.2],
    ),
    yaxis=dict(tickformat=","),
)

fig.show()

막대 그래프는 2020년부터 2025년까지의 연도별 출시 수 규모 차이를 더 직관적으로 비교할 때 사용한다.

In [6]:
bar_fig = px.bar(
    yearly_release,
    x="release_year",
    y="game_count",
    title="연도별 Steam 인디게임 출시 수 (2020~2025)",
    labels={
        "release_year": "출시 연도",
        "game_count": "게임 출시 수",
    },
    text="game_count",
)

bar_fig.update_traces(texttemplate="%{text:,}", textposition="outside")
bar_fig.update_layout(
    template="plotly_white",
    width=1000,
    height=520,
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(START_YEAR, END_YEAR + 1)),
        range=[START_YEAR - 0.2, END_YEAR + 0.2],
    ),
    yaxis=dict(tickformat=",", title="게임 출시 수"),
    uniformtext_minsize=8,
    uniformtext_mode="hide",
)

bar_fig.show()

## 5. 간단 해석용 요약 지표

아래 결과를 바탕으로 출시 경쟁이 가장 집중된 시기와 최근 추세를 해석한다.

In [7]:
peak_row = yearly_release.loc[yearly_release["game_count"].idxmax()]
recent_years = yearly_release.tail(5)

summary = pd.DataFrame(
    {
        "metric": [
            "분석 시작 연도",
            "분석 종료 연도",
            "출시 수 최대 연도",
            "최대 연도 출시 수",
            "최근 5년 평균 출시 수",
        ],
        "value": [
            yearly_release["release_year"].min(),
            yearly_release["release_year"].max(),
            int(peak_row["release_year"]),
            int(peak_row["game_count"]),
            round(recent_years["game_count"].mean(), 1),
        ],
    }
)

summary

,metric,value
0,분석 시작 연도,2020.0
1,분석 종료 연도,2025.0
2,출시 수 최대 연도,2024.0
3,최대 연도 출시 수,8855.0
4,최근 5년 평균 출시 수,6458.8


### 해석

- 출시 수가 증가하는 구간은 Steam 인디게임 시장의 공급 경쟁이 강해진 시기로 볼 수 있다.
- 출시 수가 급증한 연도에는 단순 출시만으로 노출을 확보하기 어려우므로, 장르·태그 포지셔닝과 출시 전 위시리스트 확보 전략이 더 중요하다.
- 최근 5년 평균 출시 수와 최대 출시 연도를 비교하면 현재 시장이 과거 대비 얼마나 혼잡한지 판단할 수 있다. 단, 실행일이 포함된 현재 연도는 아직 진행 중인 연도이므로 전년 대비 감소처럼 보일 수 있다.

## 6. 2023~2025년 집중 분석 근거

앞의 SteamSpy 출시 수 추이는 전체 시장 공급량을 보여준다. 2023년 이후 출시 수가 다시 증가했고 2024년에 크게 확대되었으므로, 2023~2025년은 출시 후 유저 반응 확보 난이도를 현실적으로 보여주기에 적합한 구간이다.

단, 2025년은 수집 시점 기준 진행 중인 연도이므로 완성된 연간 성과가 아니라 최신 출시작의 초기 반응 구간으로 해석한다.

In [8]:
focus_years = yearly_release.query("2023 <= release_year <= 2025").copy()
focus_years

,release_year,game_count,yoy_change,yoy_growth_rate
3,2023,6779,628.0,10.209722
4,2024,8855,2076.0,30.623986
5,2025,4203,-4652.0,-52.535291


## 7. 성과 등급 데이터 로드

시장 전체 출시량은 `steamspy_indie_games.csv`로 확인하고, 흥행 생존 난이도는 이미 성과 등급이 부여된 `steam_indie_games_graded.csv`로 확인한다.

- `steamspy_indie_games.csv`: 전체 시장 공급량과 리뷰 확보 퍼널 확인
- `steam_indie_games_graded.csv`: 2023~2025년, 리뷰 10개 이상, EA/F2P 제외 등 성과 비교가 가능한 게임의 등급 분석

In [9]:
GRADED_DATA_PATH = Path("../../data/preprocessed/steam_indie_games_graded.csv")

if not GRADED_DATA_PATH.exists():
    raise FileNotFoundError(
        "steam_indie_games_graded.csv를 찾을 수 없습니다. "
        "data/preprocessed/steam_indie_games_graded.csv 경로를 확인하세요."
    )

graded_games = pd.read_csv(GRADED_DATA_PATH)
graded_games["release_date"] = pd.to_datetime(graded_games["release_date"], errors="coerce")
graded_games["release_year"] = graded_games["release_date"].dt.year

graded_games = graded_games.query("2023 <= release_year <= 2025").copy()
graded_games["release_year"] = graded_games["release_year"].astype(int)

print(f"Loaded: {GRADED_DATA_PATH}")
print(f"분석 대상 게임 수: {len(graded_games):,}")
print(f"분석 대상 출시 연도 범위: {graded_games['release_year'].min()}~{graded_games['release_year'].max()}")

graded_games[["appid", "name", "release_year", "total_reviews", "positive_rate", "performance_grade"]].head()

Loaded: ../../data/preprocessed/steam_indie_games_graded.csv
분석 대상 게임 수: 8,730
분석 대상 출시 연도 범위: 2023~2025


,appid,name,release_year,total_reviews,positive_rate,performance_grade
0,226620,Desktop Dungeons,2023,2276,84.007030,high_high
1,251570,7 Days to Die,2024,370046,88.607633,high_high
2,252190,Defender's Quest 2: Mists of Ruin,2025,255,61.568627,mid_low
3,269770,Secrets of Grindea,2024,8270,89.334946,high_high
4,276870,Dwelvers,2023,300,64.333333,mid_low


In [10]:
GRADE_LABEL = {
    "high_high": "대흥행",
    "high_mid": "상업적 성공",
    "high_low": "호불호",
    "mid_high": "숨겨진 명작",
    "mid_mid": "평범",
    "mid_low": "외면",
    "low_high": "니치 (틈새)",
    "low_mid": "미노출",
    "low_low": "미반응",
}

GRADE_ORDER = [
    "high_high",
    "high_mid",
    "high_low",
    "mid_high",
    "mid_mid",
    "mid_low",
    "low_high",
    "low_mid",
    "low_low",
]

GRADE_COLOR = {
    "high_high": "#2d6a4f",
    "high_mid": "#52b788",
    "high_low": "#b7e4c7",
    "mid_high": "#4C72B0",
    "mid_mid": "#adb5bd",
    "mid_low": "#DD8452",
    "low_high": "#8172B2",
    "low_mid": "#e07a5f",
    "low_low": "#C44E52",
}

graded_games["performance_grade"] = pd.Categorical(
    graded_games["performance_grade"],
    categories=GRADE_ORDER,
    ordered=True,
)

## 8. 연도별 성과 등급 분포

출시 수가 많은 시장에서 실제로 어떤 등급까지 도달했는지 보기 위해, 연도별 `performance_grade` 비율을 100% 누적 막대그래프로 확인한다.

In [11]:
yearly_grade = (
    graded_games.groupby(["release_year", "performance_grade"], observed=False)
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)

yearly_total = yearly_grade.groupby("release_year")["game_count"].transform("sum")
yearly_grade["ratio"] = yearly_grade["game_count"] / yearly_total * 100
yearly_grade["grade_label"] = yearly_grade["performance_grade"].astype(str).map(GRADE_LABEL)

yearly_grade.head()

,release_year,performance_grade,game_count,ratio,grade_label
0,2023,high_high,292,9.630607,대흥행
1,2023,high_mid,70,2.308707,상업적 성공
2,2023,high_low,33,1.088391,호불호
3,2023,mid_high,639,21.075198,숨겨진 명작
4,2023,mid_mid,165,5.441953,평범


**해석:** 연도별 성과 등급 분포를 확인한다. 대흥행(high_high) 비율은 매년 한 자릿수 초반에 머물며, 저규모·고품질(low_high) 구간이 전체의 절반 이상을 차지하는 롱테일 구조가 연도에 걸쳐 일관되게 나타난다. 2025년은 데이터 수집 시점 기준 진행 중인 연도로, 수치가 연간 전체를 반영하지 않음에 유의한다.

In [12]:
grade_fig = px.bar(
    yearly_grade,
    x="release_year",
    y="ratio",
    color="performance_grade",
    color_discrete_map=GRADE_COLOR,
    category_orders={"performance_grade": GRADE_ORDER},
    title="출시 연도별 성과 등급 비율 (2023~2025)",
    labels={
        "release_year": "출시 연도",
        "ratio": "비율(%)",
        "performance_grade": "성과 등급",
    },
    hover_data={
        "grade_label": True,
        "game_count": ":,",
        "ratio": ":.1f",
    },
)

grade_fig.update_layout(
    template="plotly_white",
    barmode="stack",
    width=1000,
    height=560,
    xaxis=dict(tickmode="array", tickvals=[2023, 2024, 2025]),
    yaxis=dict(ticksuffix="%", range=[0, 100]),
    legend_title_text="성과 등급",
)

grade_fig.show()

### 해석

`대흥행`과 `상업적 성공`은 리뷰 규모와 만족도를 동시에 확보한 게임이다. 이 비율이 낮고 `미반응`, `미노출`, `외면` 비중이 크다면, 최근 인디게임 시장에서는 출시 후 충분한 반응을 얻고 살아남는 것이 어렵다고 해석할 수 있다.

## 9. 흥행 성공률과 미반응 비율

성과 등급을 의사결정에 쓰기 쉬운 지표로 묶어, 출시 후 실제로 흥행에 도달한 게임이 얼마나 적은지 확인한다.

- 흥행 성공률: `대흥행` + `상업적 성공`
- 품질 대비 확산 부족: `숨겨진 명작` + `니치 (틈새)`
- 부진/미반응 비율: `외면` + `미노출` + `미반응`

In [13]:
SUCCESS_GRADES = {"high_high", "high_mid"}
HIDDEN_QUALITY_GRADES = {"mid_high", "low_high"}
WEAK_RESPONSE_GRADES = {"mid_low", "low_mid", "low_low"}

success_summary = (
    graded_games.assign(
        is_success=graded_games["performance_grade"].astype(str).isin(SUCCESS_GRADES),
        is_hidden_quality=graded_games["performance_grade"].astype(str).isin(HIDDEN_QUALITY_GRADES),
        is_weak_response=graded_games["performance_grade"].astype(str).isin(WEAK_RESPONSE_GRADES),
    )
    .groupby("release_year", as_index=False)
    .agg(
        total_games=("appid", "nunique"),
        success_games=("is_success", "sum"),
        hidden_quality_games=("is_hidden_quality", "sum"),
        weak_response_games=("is_weak_response", "sum"),
    )
)

for count_col, rate_col in [
    ("success_games", "흥행 성공률"),
    ("hidden_quality_games", "품질 대비 확산 부족 비율"),
    ("weak_response_games", "부진/미반응 비율"),
]:
    success_summary[rate_col] = success_summary[count_col] / success_summary["total_games"] * 100

success_summary

,release_year,total_games,success_games,hidden_quality_games,weak_response_games,흥행 성공률,품질 대비 확산 부족 비율,부진/미반응 비율
0,2023,3032,362,1765,707,11.939314,58.212401,23.317942
1,2024,3818,420,2397,727,11.000524,62.781561,19.041383
2,2025,1880,194,1269,302,10.319149,67.500000,16.063830


In [14]:
rate_long = success_summary.melt(
    id_vars=["release_year"],
    value_vars=["흥행 성공률", "품질 대비 확산 부족 비율", "부진/미반응 비율"],
    var_name="지표",
    value_name="비율",
)

rate_fig = px.line(
    rate_long,
    x="release_year",
    y="비율",
    color="지표",
    markers=True,
    title="출시 연도별 흥행 성공률과 미반응 비율 (2023~2025)",
    labels={"release_year": "출시 연도", "비율": "비율(%)"},
)

rate_fig.update_traces(line_width=3, marker_size=8)
rate_fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    width=1000,
    height=520,
    xaxis=dict(tickmode="array", tickvals=[2023, 2024, 2025]),
    yaxis=dict(ticksuffix="%"),
)

rate_fig.show()

## 10. 리뷰 규모 도달 퍼널

`steam_indie_games_graded.csv`와 `steam_indie_games_silence.csv`를 합쳐 2023~2025년 공통 전처리 모집단 기준으로 리뷰 규모 도달률을 확인한다. 출시 후 각 리뷰 규모에 도달하는 게임이 얼마나 빠르게 줄어드는지 수치로 보여줌으로써, 유저 반응 확보가 얼마나 어려운지 직접적으로 드러낸다.

퍼널 단계는 배타적 구간이 아니라 누적 도달 조건으로 계산한다.

| 퍼널 단계 | 기준 |
|----------|------|
| 출시 | 출시한 게임 수 |
| 리뷰 10개 이상 도달 | `total_reviews >= 10` |
| 리뷰 50개 이상 도달 | `total_reviews >= 50` |
| 리뷰 500개 이상 도달 | `total_reviews >= 500` |

In [15]:
SILENCE_DATA_PATH = Path("../../data/preprocessed/steam_indie_games_silence.csv")

if not SILENCE_DATA_PATH.exists():
    raise FileNotFoundError(
        "steam_indie_games_silence.csv를 찾을 수 없습니다. "
        "data/preprocessed/steam_indie_games_silence.csv 경로를 확인하세요."
    )

silence_games = pd.read_csv(SILENCE_DATA_PATH)
silence_games["release_date"] = pd.to_datetime(silence_games["release_date"], errors="coerce")
silence_games["release_year"] = silence_games["release_date"].dt.year
silence_games = silence_games.query("2023 <= release_year <= 2025").copy()
silence_games["release_year"] = silence_games["release_year"].astype(int)

review_funnel_base = pd.concat(
    [
        graded_games[["appid", "name", "release_year", "total_reviews", "positive_rate", "satisfaction_grade"]],
        silence_games.assign(positive_rate=pd.NA, satisfaction_grade=pd.NA)[
            ["appid", "name", "release_year", "total_reviews", "positive_rate", "satisfaction_grade"]
        ],
    ],
    ignore_index=True,
).drop_duplicates(subset="appid")

review_funnel_base["total_reviews"] = pd.to_numeric(
    review_funnel_base["total_reviews"], errors="coerce"
).fillna(0)

FUNNEL_ORDER = [
    "출시",
    "리뷰 10개 이상 도달",
    "리뷰 50개 이상 도달",
    "리뷰 500개 이상 도달",
]

FUNNEL_DESCRIPTION = {
    "출시": "steam_indie_games_graded + steam_indie_games_silence",
    "리뷰 10개 이상 도달": "total_reviews >= 10",
    "리뷰 50개 이상 도달": "total_reviews >= 50",
    "리뷰 500개 이상 도달": "total_reviews >= 500",
}

base_total = review_funnel_base["appid"].nunique()
reviews_500_plus = review_funnel_base.loc[
    review_funnel_base["total_reviews"] >= 500
].copy()

steps = {
    "출시": base_total,
    "리뷰 10개 이상 도달": review_funnel_base.loc[
        review_funnel_base["total_reviews"] >= 10, "appid"
    ].nunique(),
    "리뷰 50개 이상 도달": review_funnel_base.loc[
        review_funnel_base["total_reviews"] >= 50, "appid"
    ].nunique(),
    "리뷰 500개 이상 도달": reviews_500_plus["appid"].nunique(),
}

funnel_summary = pd.DataFrame(
    [
        {
            "step": step,
            "step_description": FUNNEL_DESCRIPTION[step],
            "game_count": int(game_count),
            "base_total": int(base_total),
            "reach_rate": game_count / base_total * 100 if base_total else 0,
        }
        for step, game_count in steps.items()
    ]
)

funnel_summary["step"] = pd.Categorical(
    funnel_summary["step"],
    categories=FUNNEL_ORDER,
    ordered=True,
)
funnel_summary = funnel_summary.sort_values("step").reset_index(drop=True)
funnel_summary["prev_count"] = funnel_summary["game_count"].shift(1)
funnel_summary["step_conversion_rate"] = (
    funnel_summary["game_count"] / funnel_summary["prev_count"] * 100
)
funnel_summary["drop_rate"] = 100 - funnel_summary["step_conversion_rate"]
funnel_summary.loc[0, ["step_conversion_rate", "drop_rate"]] = pd.NA

funnel_summary["reach_label"] = funnel_summary["reach_rate"].map(lambda v: f"{v:.1f}%")
funnel_summary["count_label"] = funnel_summary["game_count"].map(lambda v: f"{v:,}개")
funnel_summary["drop_label"] = funnel_summary["drop_rate"].map(
    lambda v: f"▼ {v:.1f}%" if pd.notna(v) else ""
)

print()
funnel_summary


,step,step_description,game_count,base_total,reach_rate,prev_count,step_conversion_rate,drop_rate,reach_label,count_label,drop_label
0,출시,steam_indie_games_graded + steam_indie_games_s...,15406,15406,100.000000,NaN,NaN,NaN,100.0%,"15,406개",
1,리뷰 10개 이상 도달,total_reviews >= 10,8730,15406,56.666234,15406.0,56.666234,43.333766,56.7%,"8,730개",▼ 43.3%
2,리뷰 50개 이상 도달,total_reviews >= 50,3890,15406,25.249903,8730.0,44.558992,55.441008,25.2%,"3,890개",▼ 55.4%
3,리뷰 500개 이상 도달,total_reviews >= 500,1054,15406,6.841490,3890.0,27.095116,72.904884,6.8%,"1,054개",▼ 72.9%


In [16]:
funnel_summary["text_label"] = funnel_summary.apply(
    lambda row: f"{row['count_label']} ({row['reach_label']})"
    + (f"  {row['drop_label']}" if row["drop_label"] else ""),
    axis=1,
)

fig = go.Figure()
colors = px.colors.sequential.Blues_r[:len(funnel_summary)]

fig.add_trace(
    go.Bar(
        x=funnel_summary["step"].astype(str).tolist(),
        y=funnel_summary["reach_rate"].tolist(),
        text=funnel_summary["text_label"].tolist(),
        textposition="outside",
        textfont=dict(size=12, color="#333333"),
        cliponaxis=False,
        marker_color=colors,
        customdata=funnel_summary[["step_description", "game_count", "reach_rate", "step_conversion_rate", "drop_rate"]].values,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "%{customdata[0]}<br>"
            "게임 수: %{customdata[1]:,}<br>"
            "모집단 대비 도달률: %{customdata[2]:.1f}%<br>"
            "전 단계 대비 전환율: %{customdata[3]:.1f}%<br>"
            "전 단계 대비 감소율: %{customdata[4]:.1f}%<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    template="plotly_white",
    title="리뷰 규모 도달 비율",
    width=950,
    height=520,
    showlegend=False,
    xaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=FUNNEL_ORDER,
    ),
    yaxis=dict(title="도달률", ticksuffix="%", range=[0, 115]),
)

fig.show()


## 11. 종합 해석

2023~2025년은 출시 게임 수가 많고 유저 반응 확보 경쟁이 치열한 구간이다. SteamSpy 기준 출시 수는 2023년 이후 증가했고, 2024년에 크게 확대되었다.

공통 전처리 모집단 기준 리뷰 규모 도달 퍼널을 보면, 리뷰 10개 이상, 50개 이상, 500개 이상으로 갈수록 게임 수가 빠르게 줄어든다. 즉 최근 인디게임 시장에서는 출시 후 초기 반응을 확보하는 단계와, 그 반응을 더 큰 규모로 확장하는 단계가 모두 좁은 관문이다.

따라서 이후 장르·가격·태그 분석은 "어떤 속성이 좋은가"를 넘어서, "어떤 조합이 리뷰 규모와 만족도를 동시에 확보하는 데 유리한가"를 찾는 방향으로 연결한다.

## 12. 신작 출시 수 대비 무반응·반응 게임 연도별 추이

출시된 게임 중 실제로 유저 반응(리뷰 10개 이상)을 확보한 게임과 그렇지 못한 게임(침묵)이 연도별로 어떻게 분포하는지 확인한다.

- **분모:** `graded + silence` 전처리 모집단 합산 (EA·F2P·비대상 장르 필터 적용 후 기준)
- **반응 그룹:** `steam_indie_games_graded.csv` (리뷰 10개 이상)
- **침묵 그룹:** `steam_indie_games_silence.csv` (리뷰 0~9개)

분석 범위는 2023~2025년으로 한정한다. 2025년은 진행 중인 연도이므로 절대 수치보다 비율 추이에 집중한다.

In [17]:
FOCUS_START = 2023
FOCUS_END = 2025

# 연도별 전체 출시 수 (steamspy 기준, 참고용)
total_by_year = (
    release_df.query("@FOCUS_START <= release_year <= @FOCUS_END")
    .groupby("release_year", as_index=False)
    .agg(total_released=("appid", "nunique"))
)

# review_funnel_base는 graded + silence 합산 데이터 (§10에서 로드)
yearly_group = (
    review_funnel_base.query("@FOCUS_START <= release_year <= @FOCUS_END")
    .assign(
        is_silence=lambda df: df["total_reviews"] < 10,
        is_response=lambda df: df["total_reviews"] >= 10,
    )
    .groupby("release_year", as_index=False)
    .agg(
        silence_count=("is_silence", "sum"),
        response_count=("is_response", "sum"),
    )
)

trend = total_by_year.merge(yearly_group, on="release_year", how="left")
# 분모: 전처리 모집단(graded + silence) 합산
trend["base_count"] = trend["silence_count"] + trend["response_count"]
trend["silence_rate"] = trend["silence_count"] / trend["base_count"] * 100
trend["response_rate"] = trend["response_count"] / trend["base_count"] * 100

trend[["release_year", "total_released", "base_count", "response_count", "silence_count", "response_rate", "silence_rate"]]

,release_year,total_released,base_count,response_count,silence_count,response_rate,silence_rate
0,2023,6779,5097,3032,2065,59.485972,40.514028
1,2024,8855,6876,3818,3058,55.526469,44.473531
2,2025,4203,3433,1880,1553,54.762598,45.237402


In [18]:
STACK_COLOR = {
    "반응 (리뷰 ≥10개)": "#4C72B0",
    "침묵 (리뷰 <10개)": "#C44E52",
}
STACK_ORDER = ["반응 (리뷰 ≥10개)", "침묵 (리뷰 <10개)"]

trend_long = trend.melt(
    id_vars=["release_year", "base_count"],
    value_vars=["response_count", "silence_count"],
    var_name="group",
    value_name="game_count",
)
trend_long["group"] = trend_long["group"].map(
    {
        "response_count": "반응 (리뷰 ≥10개)",
        "silence_count": "침묵 (리뷰 <10개)",
    }
)
trend_long["ratio"] = trend_long["game_count"] / trend_long["base_count"] * 100

# 절대 수 누적 막대 그래프
abs_fig = px.bar(
    trend_long,
    x="release_year",
    y="game_count",
    color="group",
    color_discrete_map=STACK_COLOR,
    category_orders={"group": STACK_ORDER},
    title="연도별 신작 출시 수 대비 무반응·반응 게임 수 (2023~2025)",
    labels={"release_year": "출시 연도", "game_count": "게임 수", "group": "그룹"},
    text="game_count",
)
abs_fig.update_traces(texttemplate="%{text:,}", textposition="inside", insidetextanchor="middle")
abs_fig.update_layout(
    template="plotly_white",
    barmode="stack",
    width=1000,
    height=540,
    xaxis=dict(tickmode="array", tickvals=[2023, 2024, 2025]),
    yaxis=dict(tickformat=",", title="게임 수"),
    legend_title_text="그룹",
)
abs_fig.show()

# 비율 누적 막대 그래프
ratio_fig = px.bar(
    trend_long,
    x="release_year",
    y="ratio",
    color="group",
    color_discrete_map=STACK_COLOR,
    category_orders={"group": STACK_ORDER},
    title="연도별 신작 출시 수 대비 무반응·반응 비율 (2023~2025)",
    labels={"release_year": "출시 연도", "ratio": "비율(%)", "group": "그룹"},
    text="ratio",
)
ratio_fig.update_traces(texttemplate="%{text:.1f}%", textposition="inside", insidetextanchor="middle")
ratio_fig.update_layout(
    template="plotly_white",
    barmode="stack",
    width=1000,
    height=540,
    xaxis=dict(tickmode="array", tickvals=[2023, 2024, 2025]),
    yaxis=dict(ticksuffix="%", range=[0, 100]),
    legend_title_text="그룹",
)
ratio_fig.show()

### 해석

- **반응 비율**이 낮고 **침묵 비율**이 높을수록, 출시 후 리뷰 10개라는 최소 반응 기준조차 넘지 못한 게임이 많다는 의미다.
- 2024년에 전체 출시 수가 크게 늘었음에도 반응 게임 비율이 유지되거나 낮아졌다면, 시장 경쟁 심화로 초기 노출 확보가 더 어려워졌다고 해석할 수 있다.
- 2025년은 수집 시점 기준 진행 중인 연도이므로 절대 수 비교보다 비율 추이에 집중한다.